In [0]:
# dim org table
silver_workforce = (spark.table("silver_workforce_fte"))
silver_workforce.show(10)

nss_themes = (spark.table("nss_themes"))
nss_themes.show(10)

people_promise = (spark.table("silver_people_promise"))
people_promise.show(10)



In [0]:

from pyspark.sql.functions import col, lower,trim, row_number,lit, concat, expr
from pyspark.sql.window import Window

# 1. Workforce organisations: keep existing codes.
workforce_org = (
    silver_workforce
    .select("org_code", "org_name")
    .filter(
        col("org_code").isNotNull() &
        (trim(col("org_code")) != "") &
        (lower(trim(col("org_code"))) != "all organisations")
    )
    .withColumn("priority", lit(1))
)


# 2. Themes organisations: generate random codes.
themes_org = (
    nss_themes
    .select("org_name")
    .distinct()
    .withColumn(
        "org_code",
        concat(lit("THEME_"), expr("uuid()"))
    )
    .withColumn("priority", lit(2))
)


# 3. People Promise organisations: generate random codes.
people_promise_org = (
    people_promise
    .select("org_name")
    .distinct()
    .withColumn(
        "org_code",
        concat(lit("PP_"), expr("uuid()"))
    )
    .withColumn("priority", lit(3))
)


# Prefer workforce codes when organisation names match.
org_window = (
    Window
    .partitionBy("org_name")
    .orderBy("priority", "org_code")
)


# 4. Combine and keep one row per organisation name.
dim_org_gold = (
    workforce_org
    .unionByName(themes_org)
    .unionByName(people_promise_org)

    # Standardise names before removing duplicates.
    .withColumn("org_name", lower(trim(col("org_name"))))

    .filter(
        col("org_name").isNotNull() &
        (col("org_name") != "") &
        (col("org_name") != "all organisations")
    )

    .withColumn(
        "preferred_row",
        row_number().over(org_window)
    )
    .filter(col("preferred_row") == 1)

    .withColumn(
        "org_key",
        row_number().over(Window.orderBy("org_name"))
    )

    .select("org_key", "org_code", "org_name")
)

dim_org_gold.show(100, truncate=False)

In [0]:
# write table to gold blob
(dim_org_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "path",
        "abfss://gold@jdnhsbronze.dfs.core.windows.net/dim_organisation/"
    ) \
    .saveAsTable(
        "org_dimension"
    ))